In [45]:
from __future__ import absolute_import, division, print_function, unicode_literals

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler

from IPython.display import clear_output
from six.moves import urllib

import tensorflow.compat.v2.feature_column as fc
import tensorflow as tf
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# Importing , Splicing, and Scaling the dataset

In [46]:
df_retail = pd.read_excel("/home/whitelister/PycharmProjects/Learning AI/ML/Datasets/online+retail/Online Retail.xlsx")
df_retail.columns = [c.upper() for c in df_retail.columns]
# Encoding
df_retail = df_retail.drop(columns=["INVOICENO", "STOCKCODE"])
df_retail["INVOICEDATE"] = pd.to_datetime(df_retail["INVOICEDATE"])
df_retail["YEAR"] = df_retail["INVOICEDATE"].dt.year
df_retail["MONTH"] = df_retail["INVOICEDATE"].dt.month
df_retail["DAY"] = df_retail["INVOICEDATE"].dt.day
df_retail["HOUR"] = df_retail["INVOICEDATE"].dt.hour
df_retail["WEEKDAY"] = df_retail["INVOICEDATE"].dt.weekday
df_retail = df_retail.drop(columns=["INVOICEDATE"])
df_retail = df_retail.dropna()
df_retail["CUSTOMERID"] = df_retail["CUSTOMERID"].astype(int)
df_retail = pd.get_dummies(
    df_retail,
    columns=["DESCRIPTION", "COUNTRY"],
    dtype=int
)
# Splicing
df = df_retail.sample(frac=1, random_state=42).reset_index(drop=True)

train = df.iloc[:int(0.6 * len(df))]
valid = df.iloc[int(0.6 * len(df)):int(0.8 * len(df))]
test = df.iloc[int(0.8 * len(df)):]

In [47]:
def check_scaled(name, X):
    print(f"\n{name}")
    print(f"Shape: {X.shape}")
    print(f"Mean: {X.mean():.6f}")
    print(f"Std : {X.std():.6f}")
    print(f"Min : {X.min():.6f}")
    print(f"Max : {X.max():.6f}")

check_scaled("Training", x_train)
check_scaled("Validation", x_valid)
check_scaled("Test", x_test)


Training
Shape: (47840, 48)
Mean: 0.022971
Std : 1.093399
Min : -3.094722
Max : 107.342449

Validation
Shape: (9042, 48)
Mean: 0.000824
Std : 0.994786
Min : -2.109567
Max : 26.386448

Test
Shape: (9043, 48)
Mean: 0.000139
Std : 0.992838
Min : -2.707468
Max : 20.815216


In [48]:
def scale_dataset(dataframe, scaler=None, oversample=False, fit_scaler=False):
    # Separate features and labels
    y = dataframe["UNITPRICE"].values
    X = dataframe.drop(columns=["UNITPRICE"]).values


    # Fit only on training data
    if fit_scaler:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
    else:
        X = scaler.transform(X)

    # Oversample only training data
    if oversample:
        ros = RandomOverSampler(random_state=42)
        X, y = ros.fit_resample(X, y)

    data = np.hstack((X, y.reshape(-1, 1)))

    return data, X, y, scaler

In [49]:
# Training (fit scaler + oversample)
train, x_train, y_train, scaler = scale_dataset(
    train,
    oversample=True,
    fit_scaler=True
)

# Validation (reuse scaler)
valid, x_valid, y_valid, _ = scale_dataset(
    valid,
    scaler=scaler
)

# Test (reuse scaler)
test, x_test, y_test, _ = scale_dataset(
    test,
    scaler=scaler
)

In [50]:
def check_scaled(name, X):
    print(f"\n{name}")
    print(f"Shape: {X.shape}")
    print(f"Mean: {X.mean():.6f}")
    print(f"Std : {X.std():.6f}")
    print(f"Min : {X.min():.6f}")
    print(f"Max : {X.max():.6f}")

check_scaled("Training", x_train)
check_scaled("Validation", x_valid)
check_scaled("Test", x_test)


Training
Shape: (47840, 48)
Mean: 0.022971
Std : 1.093399
Min : -3.094722
Max : 107.342449

Validation
Shape: (9042, 48)
Mean: 0.000824
Std : 0.994786
Min : -2.109567
Max : 26.386448

Test
Shape: (9043, 48)
Mean: 0.000139
Std : 0.992838
Min : -2.707468
Max : 20.815216
